# Data Preparation for Recommender System
This notebook generates a synthetic dataset of historical cart transactions for a pizza chain and performs basic data cleansing. The output tables feed into:
* **01_market_basket_analysis** — FPGrowth-based association rules
* **02_collaborative_filter** — ALS-based user-item recommendations

### Compute Requirements
| | |
|---|---|
| **Runtime** | DBR 17.3 ML |
| **Compute** | Classic cluster (single node is sufficient) |

### Output Tables
| Table | Description |
|---|---|
| `cleaned_mapped_dataset` | Full cleaned dataset (all orders) |
| `train_dataset` | 80 % training split (seed=42) |
| `test_dataset` | 20 % held-out test split — shared across both downstream notebooks for fair evaluation |

### Cleansing Steps
1. Parse raw comma-separated order strings into arrays
2. Remove non-recommendable items (sauces, utensils)
3. Filter out empty orders

In [0]:
catalog = 'users'
schema = 'jon_cheung'

# Output table names
cleaned_table = f'{catalog}.{schema}.cleaned_mapped_dataset'
train_table   = f'{catalog}.{schema}.train_dataset'
test_table    = f'{catalog}.{schema}.test_dataset'

# Ensure the schema exists
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}')

In [0]:
import random
from pyspark.sql.types import StructType, StructField, StringType

random.seed(42)

# ---------------------
# Product catalog
# ---------------------
pizzas = [
    'pepperoni-pizza', 'cheese-pizza', 'supreme-pizza', 'meat-lovers-pizza',
    'veggie-pizza', 'hawaiian-pizza', 'bbq-chicken-pizza', 'deep-dish-pepperoni',
]
sides = ['crazy-bread', 'italian-cheese-bread', 'caesar-wings', 'stuffed-crazy-bread']
drinks = ['pepsi', 'mountain-dew', 'diet-pepsi', 'orange-fanta']
desserts = ['cookie-dough-brownie', 'cinnamon-loaded-crazy-bites']

# Items that should NOT be recommended (to be cleaned out)
sauce_list = ['marinara-sauce', 'ranch-dressing']
utensil_list = ['plastic-fork', 'plastic-knife']

# ---------------------
# Generate transactions
# ---------------------
num_users = 2_000
num_orders = 15_000
user_ids = [str(random.randint(10_000_000, 99_999_999)) for _ in range(num_users)]

orders = []
for i in range(num_orders):
    mpid = random.choice(user_ids)
    order_id = f'ORD-{i+1:06d}'

    # Every order has at least 1 pizza
    items = random.sample(pizzas, random.randint(1, 3))

    # Sides (~60 %), drinks (~50 %), desserts (~20 %)
    if random.random() < 0.60:
        items.append(random.choice(sides))
    if random.random() < 0.50:
        items.append(random.choice(drinks))
    if random.random() < 0.20:
        items.append(random.choice(desserts))

    # Sprinkle in items we will later clean out
    if random.random() < 0.35:
        items.append(random.choice(sauce_list))
    if random.random() < 0.25:
        items.append(random.choice(utensil_list))

    orders.append((mpid, order_id, ', '.join(items)))

# Build Spark DataFrame (mimics a raw orders table)
raw_schema = StructType([
    StructField('mpid', StringType()),
    StructField('order_id', StringType()),
    StructField('order_products', StringType()),   # comma-separated product names
])
sdf_raw = spark.createDataFrame(orders, raw_schema)
print(f'Generated {sdf_raw.count():,} raw orders')
display(sdf_raw.limit(10))

## Data Cleaning
Remove items we should never recommend — condiment add-ons (sauces) and disposable utensils — then drop any orders that become empty after filtering.

In [0]:
from pyspark.sql.functions import col, expr

# 1. Split comma-separated string into an array of unique, trimmed product slugs
sdf_prepared = sdf_raw.withColumn(
    'order_product_list',
    expr("array_distinct(transform(split(order_products, ','), x -> trim(x)))")
)

# 2. Define items to remove
sauce_list  = ['marinara-sauce', 'ranch-dressing']
utensil_list = ['plastic-fork', 'plastic-knife']
items_to_remove = sauce_list + utensil_list

# 3. Filter out unwanted items from each order
remove_expr = ', '.join([f"'{item}'" for item in items_to_remove])
sdf_cleaned = sdf_prepared.withColumn(
    'order_product_list',
    expr(f"filter(order_product_list, x -> NOT array_contains(array({remove_expr}), x))")
)

# 4. Drop empty strings and orders with no remaining products
sdf_cleaned = sdf_cleaned.withColumn(
    'order_product_list',
    expr("filter(order_product_list, x -> x != '')")
)
sdf_cleaned_mapped = (
    sdf_cleaned
    .filter(expr('size(order_product_list) > 0'))
    .select('mpid', 'order_id', 'order_product_list')
)

print(f'Cleaned dataset: {sdf_cleaned_mapped.count():,} orders')
display(sdf_cleaned_mapped.limit(10))

In [0]:
# Save the full cleaned dataset (used by both 01_MBA and 02_collab_filter)
sdf_cleaned_mapped.write.format('delta').mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable(cleaned_table)

# 80/20 train-test split — both downstream notebooks load these directly
# so they evaluate against the exact same held-out set
train, test = sdf_cleaned_mapped.randomSplit([0.8, 0.2], seed=42)

train.write.format('delta').mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable(train_table)

test.write.format('delta').mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable(test_table)

print(f'Saved to {catalog}.{schema}:')
print(f'  {cleaned_table}  {sdf_cleaned_mapped.count():,} orders')
print(f'  {train_table}           {train.count():,} orders')
print(f'  {test_table}            {test.count():,} orders')